# Qversity Fintech — Análisis de Preguntas de Negocio

**Pipeline:** ELT migrado a Databricks Free Edition · **Dataset:** ~5.100 clientes, 7 países LATAM, 87.686 transacciones  
**Fuente:** `workspace.gold.*` (Unity Catalog)

| # | Pregunta | Dashboard | Mart |
|---|---|---|---|
| Q9 | ¿El risk score predice la delinquencia? | Risk & Credit | `mart_risk_buckets` |
| Q18 | ¿Cuál es la tasa de fallo por canal de transacción? | Revenue & Transactions | `mart_tx_by_channel` |
| Q12 | ¿Hay estacionalidad en la adquisición de clientes? | Customer & Engagement | `mart_acquisition_trend` |

In [0]:
CATALOG = "workspace"
spark.sql(f"USE CATALOG {CATALOG}")

DataFrame[]

---
## Q9 — ¿El risk score predice la delinquencia?

**Definición de negocio:** Los clientes se segmentan en 4 buckets por `risk_score` (0-100):
`low / medium / high / critical`. En un modelo de scoring bien calibrado, la tasa de delinquencia
debería subir monotónicamente de `low` a `critical`. Si la curva es plana o invertida,
el score no tiene poder predictivo — y una institución que lo use para asignar capital
estaría cometiendo un error sistemático.

**Resultado esperado (4 filas):**

| risk_bucket | customer_count | share | observed_delinquency_rate |
|---|---|---|---|
| low | 1495 | 0.2990 | **0.6533** |
| medium | 1539 | 0.3078 | 0.6324 |
| high | 1218 | 0.2436 | 0.6277 |
| critical | 748 | 0.1496 | **0.6043** |

In [0]:
df_q9 = spark.sql("""
    SELECT
        risk_bucket,
        customer_count,
        ROUND(bucket_share, 4)      AS share_of_customers,
        ROUND(delinquency_rate, 4)  AS observed_delinquency_rate,
        ROUND(default_rate, 4)      AS default_rate
    FROM gold.mart_risk_buckets
    ORDER BY
        CASE risk_bucket
            WHEN 'low'      THEN 1
            WHEN 'medium'   THEN 2
            WHEN 'high'     THEN 3
            WHEN 'critical' THEN 4
        END
""")
display(df_q9)

risk_bucket,customer_count,share_of_customers,observed_delinquency_rate,default_rate
low,1495,0.2990,0.6533,0.4356
medium,1539,0.3078,0.6324,0.4288
high,1218,0.2436,0.6277,0.4301
critical,748,0.1496,0.6043,0.4155


**Hallazgo — el más importante del proyecto:**

La curva de delinquencia es **invertida**: los clientes `low` risk muestran 65.3% de delinquencia,
mientras los `critical` muestran 60.4%. En un modelo bien calibrado debería ser al revés.

Esto significa que el campo `risk_score` del dataset **no está correlacionado** con el estado
real de los préstamos — ambos fueron generados independientemente por el generador sintético.
En producción, un score anti-predictivo es peor que no tener score: asigna capital y esfuerzo
de cobranza hacia los clientes equivocados.

Documentado en `decisions.md` bajo "Key findings §1".

---
## Q18 — ¿Cuál es la tasa de fallo por canal de transacción?

**Definición de negocio:** `failed_rate = failed_tx_count / tx_count` (sobre todas las transacciones
intentadas, no solo las completadas). El benchmark real de la industria es < 3%.
Un canal con alta tasa de fallo implica pérdida de ingresos, fricción en la experiencia del cliente
y costos de resolución de disputas.

**Resultado esperado (5 filas):**

| channel | total_tx | failed_tx | failed_rate |
|---|---|---|---|
| atm | 17134 | 4316 | 0.2519 |
| branch | 17077 | 4295 | 0.2515 |
| pos | 17023 | 4253 | 0.2498 |
| mobile | 17092 | 4245 | 0.2484 |
| web | 16741 | 4138 | 0.2472 |

In [0]:
df_q18 = spark.sql("""
    SELECT
        channel,
        SUM(tx_count)           AS total_tx,
        SUM(completed_tx_count) AS completed_tx,
        SUM(failed_tx_count)    AS failed_tx,
        ROUND(
            SUM(failed_tx_count) * 1.0 / NULLIF(SUM(tx_count), 0), 4
        )                       AS failed_rate,
        ROUND(
            SUM(completed_tx_count) * 1.0 / NULLIF(SUM(tx_count), 0), 4
        )                       AS completion_rate,
        ROUND(AVG(avg_ticket), 2) AS avg_ticket_usd
    FROM gold.mart_tx_by_channel
    GROUP BY channel
    ORDER BY failed_rate DESC
""")
display(df_q18)

channel,total_tx,completed_tx,failed_tx,failed_rate,completion_rate,avg_ticket_usd
pos,17552,4362,4419,0.2518,0.2485,1.796346596E7
branch,17583,4436,4421,0.2514,0.2523,1.888495682E7
atm,17679,4351,4427,0.2504,0.2461,7951617.31
mobile,17611,4449,4375,0.2484,0.2526,2969543.71
web,17261,4309,4273,0.2476,0.2496,1.146909908E7


**Hallazgo:**

La tasa de fallo ronda el **25% en todos los canales** — ~8x el benchmark real de la industria.
Los cinco canales son prácticamente indistinguibles entre sí (rango: 24.7% a 25.2%), lo que
confirma que el generador distribuyó los estados de transacción de forma uniforme entre
`completed / pending / failed / reversed` (≈25% cada uno), sin correlación con el canal.

En producción, esta uniformidad sería una señal de alerta sistémica: si todos los canales
fallan igual, el problema no está en el canal sino en la infraestructura transaccional central.
Documentado en `decisions.md` bajo "Key findings §3".

---
## Q12 — ¿Hay estacionalidad en la adquisición mensual de clientes?

**Definición de negocio:** Nuevos clientes registrados por mes calendario, con crecimiento
acumulado y variación MoM. Si hay picos estacionales consistentes, la estrategia de marketing
debería concentrar inversión en esos meses. Si la curva es plana, la adquisición es insensible
a la estación y conviene una estrategia sostenida en lugar de campañas estacionales.

**Resultado esperado:** 73 filas (mayo 2020 → mayo 2026), promedio ~68 clientes/mes.

In [0]:
df_q12 = spark.sql("""
    SELECT
        month_label,
        new_customers,
        cumulative_customers,
        ROUND(mom_growth_pct, 2) AS mom_growth_pct,
        -- Promedio móvil 3 meses para suavizar la volatilidad
        ROUND(AVG(new_customers) OVER (
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1)                    AS rolling_avg_3m
    FROM gold.mart_acquisition_trend
    ORDER BY month
""")
display(df_q12)

month_label,new_customers,cumulative_customers,mom_growth_pct,rolling_avg_3m
2020-05,60,60,null,60.0
2020-06,63,123,5.0,61.5
2020-07,77,200,22.22,66.7
2020-08,67,267,-12.99,69.0
2020-09,71,338,5.97,71.7
2020-10,68,406,-4.23,68.7
2020-11,61,467,-10.29,66.7
2020-12,72,539,18.03,67.0
2021-01,70,609,-2.78,67.7
2021-02,61,670,-12.86,67.7


**Hallazgo:**

La adquisición mensual oscila entre **50 y 92 clientes** sin ningún patrón estacional detectable.
El promedio es estable en ~68 clientes/mes a lo largo de 6 años, sin tendencia de crecimiento
interanual (los totales anuales rondan los 820 clientes/año).

En una institución real en LATAM esperaríamos:
- **Enero:** pico por resoluciones de año nuevo y desembolso de bonos
- **Noviembre-Diciembre:** pico por campañas de fin de año
- **Julio-Agosto:** valle por vacaciones

La ausencia de estos patrones confirma que `registration_date` fue generada aleatoriamente
uniforme. El punto más llamativo es **mayo 2026: solo 17 clientes** vs el promedio de 68 —
no es una caída real sino el artefacto del momento de corte del dataset (mes incompleto).
Este mismo artefacto motivó el fix de `date_trunc('month', current_date)` en los marts
de revenue. Documentado en `decisions.md`.

---
## Resumen

| Pregunta | Resultado real | Resultado esperado en producción |
|---|---|---|
| Q9 — Risk score predice delinquencia | **Invertido**: low > critical | Monotónicamente creciente |
| Q18 — Tasa de fallo por canal | **~25% uniforme** en todos | < 3%, con variación por canal |
| Q12 — Estacionalidad en adquisición | **Plana**, sin patrón estacional | Picos en enero y Q4 |

Los tres hallazgos confirman lo documentado en `decisions.md`: el generador sintético no
correlacionó las variables financieras entre sí. El pipeline captura y expone fielmente
esa realidad — lo que demuestra que los 434 tests de dbt detectarían anomalías reales en producción.